In [1]:
import numpy
import pandas
import os
import numpy
import scipy
import mod_utils as utils
import matplotlib.pyplot as plt
import os

def attempt_to_print_halfwidth(list1):
    y_min = []
    y_max = []
    val = []
    time = []
    for idx, list_item in enumerate(list1.df_list):
        # Assuming there's a column named 'time' that defines the time axis
        # If not present, replace it with appropriate logic to define time.

        try:
            #print('reading frame: ', list1.timesteps[idx])
            for i in range(1, len(list_item["min_mixWidth"])):
                check_val = list_item["min_mixWidth"][i-1]*list_item["min_mixWidth"][i]
                if check_val < 0:
                    min_idx = i
            for i in range(1, len(list_item["max_mixWidth"])):
                check_val = list_item["max_mixWidth"][i-1]*list_item["max_mixWidth"][i]
                if check_val < 0:
                    max_idx = i   
            min_idx2 = numpy.where(list_item["min_mixWidth"][:-1] * list_item["min_mixWidth"][1:] < 0)[0][0] + 1
            max_idx2 = numpy.where(list_item["max_mixWidth"][:-1] * list_item["max_mixWidth"][1:] < 0)[0][0] + 1
            #print(min_idx, min_idx2)
            #print(max_idx, max_idx2)
            y_min.append(list_item["y_dat"][min_idx])
            y_max.append(list_item["y_dat"][max_idx])
            diff = float((y_max[-1]-y_min[-1]))/(2*list1.wavelength)
            val.append(diff)
            time.append(list1.timesteps[idx])#*nonDim_T)
        except IndexError:
            print(f"Sign change not found in DataFrame {idx}. Skipping.")
            continue

    plt.figure(figsize=(10, 6))
    plt.scatter(time, val, label='', marker='o')
    #plt.scatter(list1.timesteps, y_max, label='Max Position', marker='x')

    plt.xlabel('Time')
    plt.ylabel('y_dat Value')
    plt.title('Plot of y_dat at Min and Max Positions Over Time')
    plt.legend()
    plt.grid(True)

    plt.show()

In [2]:
math_vars = {
    'high_rho': 1.5,
    'low_rho' : 1.0,
    'high_visc' : 0.0005, #dynamic
    'low_visc' : 0.0005,#0.0005,
    'wavelength' : 1,
    'grev': 1,
    'x_L' : 1,
    'y_L' : 4,
    'delta_h': 0.0045,
    'timestep': 0.0035,
    'time_end': 5,
    'maxWidthCutOff': 0.01,
    'surface_tension': 0.0001
}

model = utils.dynamic_model(math_vars)
model.set_nondim_consts(ref_l = 'x_L',
                        ref_rho = 'high_rho',
                        ref_grev = 'grev',
                        ref_visc = 'low_visc',
                        ref_visc_kin = lambda ref_visc, ref_rho: ref_visc/ref_rho,
                        ref_u = lambda ref_grev, ref_l: numpy.sqrt(ref_grev*ref_l),
                        ref_surface_tension = lambda ref_l, ref_rho, ref_u, surface_tension: ref_rho*ref_l*numpy.pow(ref_u, 2)*surface_tension
                        )

model.calculate_nondim_nums(atwood = {},
                            reynolds = {},
                            froude = {},
                            grashof = {},
                            #ren2 = lambda ref_grev, ref_l, ref_rho, ref_visc: (numpy.sqrt(numpy.pow(ref_l,3)*ref_grev)*ref_rho)/ref_visc
                            weber = lambda ref_rho, ref_l, ref_u, ref_surface_tension: ref_rho*ref_l*(numpy.pow(ref_u, 2))/ref_surface_tension,
                            eotvos = lambda high_rho, low_rho, ref_grev, ref_l, ref_surface_tension: (high_rho - low_rho)*ref_grev*numpy.pow(ref_l, 2)/ref_surface_tension
                            )

{'atwood_num': 0.2, 'reynolds_num': np.float64(3000.0), 'froude_num': np.float64(1.0), 'grashof_num': np.float64(0.32804999999999995), 'weber_num': np.float64(9999.999999999998), 'eotvos_num': np.float64(3333.333333333333)}


In [3]:
name_vars = {
    'x_dat': "X (m)",
    'y_dat': "Y (m)",
    'v_x': "Velocity[i] (m/s)",
    'v_y': "Velocity[j] (m/s)",
    'rel_v_x': "Relative Velocity[i] (m/s)",
    "rel_v_y": "Relative Velocity[j] (m/s)",
    'volFrac_high': "Volume Fraction of high_rho",
    'volFrac_low': "Volume Fraction of low_rho",
    'min_mixWidth': "minMixWidthEval",
    'max_mixWidth': "maxMixWidthEval",
    'massFlux': "Report: massImbalance_highRho (kg)",
    'mean_rho': "Density (kg/m^3)"
}


dir = r'D:\thesis\rayleigh_taylor\test'
name = 'x_mid_L_test_'
name_append = '.csv'

ref_t = lambda atwood_num: numpy.sqrt(atwood_num)

model.load_solution_data(dir, name, name_append, name_vars, ref_t= ref_t)

2 0.0035


In [41]:
for i in range(0, 2200):
    time = i*0.0055
    if time >= 6.99 and time < 7.01:
        print(i)
    
    




1271
1272
1273
1274


In [ ]:
y_s = []
y_b = []
v_y_s = []
v_y_b = []
val = []
alpha = []
time = []
nondimtime = []
h_0 = 0
for idx, working_df in enumerate(model.solution_data):
    min_val = working_df["y_dat"].min()
    max_val = working_df["y_dat"].max()
    y_s.append(min_val)
    y_b.append(max_val)
    h_t = (max_val - min_val)/2
    if idx == 0:
        h_0 = h_t
    #print(h_t, h_0)
    val.append(h_t)
    time.append(model.params['timesteps'][idx])
    nondimtime.append(model.params['nondim_time_time'][idx])
    #sqrt(ht^0.5 - h_(0)^(0.5)/((Ag)^(0.5)*(t-t0))
    if idx > 0:
        val2 = numpy.sqrt((numpy.sqrt(h_t) - numpy.sqrt(h_0)/(numpy.sqrt(model.params['atwood_num'] * numpy.abs(model.params['grev']))*(model.params['timesteps'][idx] - model.params['timesteps'][0]))))
    else:
        val2 = 0
    alpha.append(val2)
    min_idx = numpy.argmin(working_df["y_dat"])
    max_idx = numpy.argmax(working_df["y_dat"])
    #v_y_b.append((working_df["v_y"][max_idx]/outlist.FroudeConst))
    v_y_s.append(working_df["v_y"][min_idx])

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(nondimtime, v_y_b, label='val', color='green')
plt.xlabel("Non-dimensional Time")
plt.ylabel("Value (val)")
plt.title("val vs Non-dimensional Time")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Plot y_max and y_min on the same graph
plt.figure(figsize=(8, 5))
plt.plot(nondimtime, y_b, label='y_max', marker = ".")
plt.plot(nondimtime, y_s, label='y_min', marker = '.')
plt.xlabel("Non-dimensional Time")
plt.ylabel("y_min and y_max")
plt.title("y_min and y_max vs Non-dimensional Time")
plt.legend()
plt.grid(True)
plt.show()

# Plot val against nondimtime
plt.figure(figsize=(8, 5))
plt.plot(nondimtime, val, label='val', color='green', marker = '.')
plt.xlabel("Non-dimensional Time")
plt.ylabel("Value (val)")
plt.title("val vs Non-dimensional Time")
plt.legend()
plt.grid(True)
plt.show()